In [ ]:
import numpy as np
import re
import os
import joblib
from Bio import SeqIO
from itertools import product
from collections import Counter
from sklearn.model_selection import StratifiedKFold, GridSearchCV
#from sklearn.svm import SVC
from cuml.svm import SVC  # GPU-accelerated SVC
from sklearn.metrics import (
    roc_curve, auc, average_precision_score, accuracy_score,
    recall_score, f1_score, matthews_corrcoef, confusion_matrix
)

In [ ]:
# --- Step 1: Load sequences ---
print("Loading sequences...")
pos_seq = [str(seq_record.seq).upper() for seq_record in SeqIO.parse('../Datasets/GSM3680571_72_73_Pos.txt', "fasta")]
neg_seq = [str(seq_record.seq).upper() for seq_record in SeqIO.parse('../Datasets/Negative_3.txt', "fasta")]
neg_seq = neg_seq[:len(pos_seq)]

combine_seqs = pos_seq + neg_seq
labels = np.array([1] * len(pos_seq) + [0] * len(neg_seq))

In [ ]:
# --- Step 2: Extract k-mer features ---
# Generate all possible k-mers from DNA alphabet
def generate_kmer_patterns(k):
    alphabet = ['A', 'C', 'G', 'T']
    return [''.join(p) for p in product(alphabet, repeat=k)]

# Tokenize a single sequence into overlapping k-mers
def generate_kmers(sequence, k=5):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

# Convert tokenized k-mers into normalized count vectors based on patterns
def get_kmer_feature_vectors(sequences, k):
    patterns = generate_kmer_patterns(k)
    features = []
    for seq in sequences:
        kmers = generate_kmers(seq, k)
        counts = Counter(kmers)
        total_kmers = len(kmers)
        if total_kmers == 0:
            feature_vector = [0.0 for _ in patterns]
        else:
            feature_vector = [counts.get(pat, 0) / total_kmers for pat in patterns]
        features.append(feature_vector)
    return np.array(features)

# Example usage
k = 5
features = get_kmer_feature_vectors(combine_seqs, k)
print(features.shape)
#print(features)


In [ ]:
# --- Create fold indices ---
fold_indices_path = "fold_indices.joblib"
if os.path.exists(fold_indices_path):
    print(" Loading saved fold indices...")
    fold_indices = joblib.load(fold_indices_path)
else:
    print("Creating and saving new fold indices...")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_indices = [(train_idx, test_idx) for train_idx, test_idx in skf.split(features, labels)]
    joblib.dump(fold_indices, fold_indices_path)
    print(f" Saved fold indices to {fold_indices_path}")

# Create directories
os.makedirs("fold_data", exist_ok=True)
os.makedirs("fold_models", exist_ok=True)


In [ ]:
# --- Metric containers ---
metrics = {
    'Accuracy': [], 'Sensitivity': [], 'Specificity': [],
    'MCC': [], 'F1': [], 'AUC_ROC': [], 'AUC_PR': []
}

# --- Cross-validation loop ---
for fold, (train_idx, test_idx) in enumerate(fold_indices, 1):
    print(f"\n Fold {fold}")

    X_train, X_test = features[train_idx], features[test_idx]
    y_train, y_test = labels[train_idx], labels[test_idx]

    # Save fold data
    fold_data = {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }
    joblib.dump(fold_data, f"fold_data/fold{fold}_data.joblib")

    # Train SVM
    model = SVC(C=10, gamma='scale', kernel='rbf', probability=True)
    model.fit(X_train, y_train)

    # Save model for this fold
    model_path = f"fold_models/svm_model_fold{fold}.joblib"
    joblib.dump(model, model_path)

    # Evaluate
    y_pred = model.predict(X_test)
    y_scores = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    sen = recall_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spe = tn / (tn + fp)
    mcc = matthews_corrcoef(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_roc = auc(*roc_curve(y_test, y_scores)[:2])
    auc_pr = average_precision_score(y_test, y_scores)

    metrics['Accuracy'].append(acc)
    metrics['Sensitivity'].append(sen)
    metrics['Specificity'].append(spe)
    metrics['MCC'].append(mcc)
    metrics['F1'].append(f1)
    metrics['AUC_ROC'].append(auc_roc)
    metrics['AUC_PR'].append(auc_pr)

    print(f" Fold {fold} - Accuracy: {acc:.4f}, Sensitivity: {sen:.4f}, Specificity: {spe:.4f}, "
          f"MCC: {mcc:.4f}, F1: {f1:.4f}, AUC-ROC: {auc_roc:.4f}, AUC-PR: {auc_pr:.4f}")

# --- Average performance ---
print("\n === Average Metrics Across All Folds ===")
for key in metrics:
    print(f"{key}: {np.mean(metrics[key]):.4f} ± {np.std(metrics[key]):.4f}")


In [ ]:
# --- Step 5: Report average performance ---
print("\n === Average Metrics Across 5 Folds ===")
for key in metrics:
    mean = np.mean(metrics[key])
    std = np.std(metrics[key])
    print(f"{key}: {mean:.4f} ± {std:.4f}")

print("\n Best model saved as: best_svm_model.joblib")
print(" All fold data saved in: fold_data/")


In [ ]:
import pandas as pd
import numpy as np

# --- Step 5: Report average performance ---
print("\n === Average Metrics Across 5 Folds ===")
formatted_metrics = {}

for key in metrics:
    mean = np.mean(metrics[key])
    std = np.std(metrics[key])
    formatted = f"{mean:.4f} ± {std:.4f}"
    formatted_metrics[key] = formatted
    print(f"{key}: {formatted}")

# Save the metrics to an Excel file in the desired format
df_metrics = pd.DataFrame([formatted_metrics])  # Create a one-row DataFrame
df_metrics.to_excel("average_metrics.xlsx", index=False, engine='openpyxl')

In [ ]:
### Access the save model 

In [ ]:
import os
import joblib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, roc_curve, auc, average_precision_score
)

num_folds = 5  # or however many folds you used

metrics = {
    'Accuracy': [], 'Sensitivity': [], 'Specificity': [],
    'MCC': [], 'F1': [], 'AUC_ROC': [], 'AUC_PR': []
}

plt.figure(figsize=(10, 8))

for fold in range(1, num_folds + 1):
    print(f"\n Evaluating Fold {fold} from saved model/data")

    # Load model and data
    model = joblib.load(f"fold_models/svm_model_fold{fold}.joblib")
    data = joblib.load(f"fold_data/fold{fold}_data.joblib")

    X_test, y_test = data['X_test'], data['y_test']
    y_pred = model.predict(X_test)
    y_scores = model.predict_proba(X_test)[:, 1]

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    sen = recall_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spe = tn / (tn + fp)
    mcc = matthews_corrcoef(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    auc_roc = auc(fpr, tpr)
    auc_pr = average_precision_score(y_test, y_scores)

    # Store metrics
    metrics['Accuracy'].append(acc)
    metrics['Sensitivity'].append(sen)
    metrics['Specificity'].append(spe)
    metrics['MCC'].append(mcc)
    metrics['F1'].append(f1)
    metrics['AUC_ROC'].append(auc_roc)
    metrics['AUC_PR'].append(auc_pr)

    # Plot ROC
    plt.plot(fpr, tpr, label=f"Fold {fold} (AUC = {auc_roc:.2f})")

    print(f" Fold {fold} - Acc: {acc:.4f}, Sens: {sen:.4f}, Spec: {spe:.4f}, "
          f"MCC: {mcc:.4f}, F1: {f1:.4f}, AUC-ROC: {auc_roc:.4f}, AUC-PR: {auc_pr:.4f}")

# ROC Plot formatting
plt.plot([0, 1], [0, 1], 'k--', label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for All Folds")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("roc_all_folds.png")
plt.show()

# Print averaged metrics
print("\n === Averaged Metrics ===")
for key in metrics:
    print(f"{key}: {np.mean(metrics[key]):.4f} ± {np.std(metrics[key]):.4f}")


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import joblib

# Load model and data
fold = 1
model = joblib.load(f"fold_models/svm_model_fold{fold}.joblib")
data = joblib.load(f"fold_data/fold{fold}_data.joblib")

X_train = data['X_train']
y_train = data['y_train']

# Use predicted probabilities (2D output)
svm_outputs = model.predict_proba(X_train)

# Run t-SNE
print(" Running t-SNE on predicted probabilities...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(svm_outputs)

# Plot
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_train, cmap='viridis', alpha=0.8, s=5)
plt.title("t-SNE of SVM Predict Probabilities (Fold 1 Training Set)")
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.legend(*scatter.legend_elements(), title="Classes")
plt.grid(True)
plt.tight_layout()
plt.savefig("tsne_svm_proba_fold1_train.png")
plt.show()
